# NeXo v3.0 — CEM Experience Score Model Training

## Methodology: MLOps Model Development Lifecycle

This notebook implements **Phase 4-6** of the MLOps lifecycle:
- **Phase 4**: Train/Validate/Test Split (stratified + temporal)
- **Phase 5**: Model Training & Hyperparameter Tuning
- **Phase 6**: Evaluation & Explainability

### Why LightGBM over GBR for v3.0?
| Criterion | GradientBoosting (v2.0) | LightGBM (v3.0) |
|-----------|------------------------|-----------------|
| Training speed | Slow (sklearn native) | **10-50× faster** (histogram-based) |
| Memory usage | High | **Lower** (GOSS + EFB) |
| Large datasets | Struggles >100K | **Handles millions** |
| GPU support | No | **Yes** |
| Interpretability | `feature_importances_` | Built-in + SHAP native |
| Academic defense | Simpler to explain | State-of-art, well-cited |

### Strategy
1. **Baseline**: Retrain GBR on real data (fair comparison with v2.0)
2. **Candidate**: LightGBM with RandomizedSearchCV
3. **Winner selection**: Best CV R² on held-out test set
4. **Explainability**: SHAP values for top features
5. **Artifact**: Save winning model + feature importance plot

In [ ]:
# --- Phase 0: Imports ---
import os
import warnings
from datetime import datetime

import joblib
import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import shap
from scipy import stats
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import RandomizedSearchCV, train_test_split

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

SEED = 42
np.random.seed(SEED)
ARTIFACT_DIR = "data"
MODEL_DIR = "models"

print(f"[{datetime.now():%H:%M:%S}] CEM v3.0 Training Started")
print(f"Packages: LightGBM {lgb.__version__}, SHAP {shap.__version__}")

## Phase 4: Load Dataset & Train/Test Split
We use the real-data CEM dataset from Notebook 05 (500K subscribers, March 2026).

In [ ]:
# Load NPZ
cem_data = np.load(f"{ARTIFACT_DIR}/cem_training_mar2026.npz", allow_pickle=True)
X = cem_data["X"]
y = cem_data["y"]
feature_names = list(cem_data["feature_names"])

print(f"Dataset loaded: X={X.shape}, y={y.shape}")
print(f"Features: {feature_names}")
print(f"Target stats: mean={y.mean():.4f}, std={y.std():.4f}, min={y.min():.4f}, max={y.max():.4f}")

# Train/val/test split: 70/15/15
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=SEED
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=SEED
)

print(f"\nSplit sizes:")
print(f"  Train: {len(y_train):,} ({len(y_train)/len(y)*100:.1f}%)")
print(f"  Val:   {len(y_val):,} ({len(y_val)/len(y)*100:.1f}%)")
print(f"  Test:  {len(y_test):,} ({len(y_test)/len(y)*100:.1f}%)")

## Phase 5A: Baseline — GradientBoostingRegressor (v2.0 Architecture)
We retrain the v2.0 model on real data to establish a fair baseline.

In [ ]:
print("=" * 60)
print("Training Baseline: GradientBoostingRegressor")
print("=" * 60)

gbr = GradientBoostingRegressor(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.1,
    random_state=SEED,
    verbose=1,
)

gbr.fit(X_train, y_train)

# Evaluate
y_pred_gbr_val = gbr.predict(X_val)
y_pred_gbr_test = gbr.predict(X_test)

def report(name, y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    print(f"\n{name}")
    print(f"  R²   = {r2:.4f}")
    print(f"  MAE  = {mae:.4f}")
    print(f"  RMSE = {rmse:.4f}")
    return {"r2": r2, "mae": mae, "rmse": rmse}

metrics_gbr_val = report("GBR Validation", y_val, y_pred_gbr_val)
metrics_gbr_test = report("GBR Test", y_test, y_pred_gbr_test)

### Baseline Feature Importances

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
importance = pd.Series(gbr.feature_importances_, index=feature_names).sort_values()
importance.plot(kind="barh", ax=ax, color="steelblue")
ax.set_title("GBR Feature Importances (Baseline)")
ax.set_xlabel("Importance")
plt.tight_layout()
plt.savefig(f"{ARTIFACT_DIR}/gbr_feature_importance.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {ARTIFACT_DIR}/gbr_feature_importance.png")

## Phase 5B: Candidate — LightGBM with Hyperparameter Tuning
LightGBM uses histogram-based algorithms and GOSS (Gradient-based One-Side Sampling)  
for orders-of-magnitude speedup on large tabular data.

In [ ]:
print("=" * 60)
print("Training Candidate: LightGBM")
print("=" * 60)

lgb_train = lgb.Dataset(X_train, label=y_train)
lgb_val = lgb.Dataset(X_val, label=y_val, reference=lgb_train)

# Base params
base_params = {
    "objective": "regression",
    "metric": "rmse",
    "boosting_type": "gbdt",
    "verbosity": -1,
    "random_state": SEED,
    "n_jobs": -1,
}

# Phase 1: Quick coarse search with early stopping
print("\n[Phase 5B.1] Coarse hyperparameter search...")

coarse_grid = {
    "num_leaves": [31, 63, 127],
    "learning_rate": [0.05, 0.1, 0.2],
    "feature_fraction": [0.8, 1.0],
    "bagging_fraction": [0.8, 1.0],
    "bagging_freq": [5],
    "min_child_samples": [20, 50],
}

best_score = float("inf")
best_params = None

for num_leaves in coarse_grid["num_leaves"]:
    for lr in coarse_grid["learning_rate"]:
        for feat_frac in coarse_grid["feature_fraction"]:
            params = {
                **base_params,
                "num_leaves": num_leaves,
                "learning_rate": lr,
                "feature_fraction": feat_frac,
                "bagging_fraction": 0.8,
                "bagging_freq": 5,
                "min_child_samples": 20,
            }
            model = lgb.train(
                params,
                lgb_train,
                num_boost_round=500,
                valid_sets=[lgb_val],
                callbacks=[lgb.early_stopping(20, verbose=False)],
            )
            score = model.best_score["valid_0"]["rmse"]
            if score < best_score:
                best_score = score
                best_params = params.copy()
                best_params["num_boost_round"] = model.best_iteration

print(f"Best coarse RMSE: {best_score:.6f}")
print(f"Best params: {best_params}")

### Train Final LightGBM with Best Params

In [ ]:
print("\n[Phase 5B.2] Training final LightGBM with best hyperparameters...")

final_params = {k: v for k, v in best_params.items() if k != "num_boost_round"}
final_model = lgb.train(
    final_params,
    lgb_train,
    num_boost_round=best_params["num_boost_round"],
    valid_sets=[lgb_train, lgb_val],
    callbacks=[lgb.log_evaluation(period=50)],
)

# Evaluate
y_pred_lgb_val = final_model.predict(X_val, num_iteration=final_model.best_iteration)
y_pred_lgb_test = final_model.predict(X_test, num_iteration=final_model.best_iteration)

metrics_lgb_val = report("LightGBM Validation", y_val, y_pred_lgb_val)
metrics_lgb_test = report("LightGBM Test", y_test, y_pred_lgb_test)

## Phase 6A: Model Comparison

In [ ]:
comparison = pd.DataFrame({
    "GBR (v2.0 baseline)": [metrics_gbr_test["r2"], metrics_gbr_test["mae"], metrics_gbr_test["rmse"]],
    "LightGBM (v3.0)": [metrics_lgb_test["r2"], metrics_lgb_test["mae"], metrics_lgb_test["rmse"]],
}, index=["R²", "MAE", "RMSE"])

print("\n" + "=" * 60)
print("MODEL COMPARISON (Test Set)")
print("=" * 60)
print(comparison.round(4))

# Determine winner
winner = "LightGBM" if metrics_lgb_test["r2"] > metrics_gbr_test["r2"] else "GBR"
print(f"\nWinner: {winner}")

# Save comparison
comparison.to_csv(f"{ARTIFACT_DIR}/cem_model_comparison.csv")
print(f"Saved: {ARTIFACT_DIR}/cem_model_comparison.csv")

## Phase 6B: Residual Analysis
Good regression models should have residuals that are normally distributed around zero.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Prediction vs Actual
axes[0].scatter(y_test, y_pred_lgb_test, alpha=0.3, s=1, color="steelblue")
axes[0].plot([0, 1], [0, 1], "r--", lw=2)
axes[0].set_xlabel("Actual CEM Score")
axes[0].set_ylabel("Predicted CEM Score")
axes[0].set_title("LightGBM: Predicted vs Actual")
axes[0].set_xlim(0, 0.8)
axes[0].set_ylim(0, 0.8)

# Residual distribution
residuals = y_test - y_pred_lgb_test
axes[1].hist(residuals, bins=100, color="green", alpha=0.7, edgecolor="black")
axes[1].set_title("Residual Distribution")
axes[1].set_xlabel("Residual (Actual - Predicted)")
axes[1].axvline(0, color="red", linestyle="--")

# Q-Q plot for normality check
stats.probplot(residuals, dist="norm", plot=axes[2])
axes[2].set_title("Q-Q Plot (Normality Check)")

plt.tight_layout()
plt.savefig(f"{ARTIFACT_DIR}/cem_residual_analysis.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {ARTIFACT_DIR}/cem_residual_analysis.png")

# Shapiro-Wilk on sample (full set too large)
_, p_value = stats.shapiro(residuals[:5000])
print(f"\nShapiro-Wilk p-value (n=5000 sample): {p_value:.6f}")
print("Interpretation: p > 0.05 → residuals are approximately normal")

## Phase 6C: SHAP Explainability
SHAP (SHapley Additive exPlanations) provides game-theoretic feature importance  
that is more reliable than native `feature_importances_` for non-linear models.

In [ ]:
print("=" * 60)
print("Computing SHAP Values (sample of 2000 for speed)")
print("=" * 60)

# Use TreeExplainer for LightGBM (fast, exact)
explainer = shap.TreeExplainer(final_model)
shap_sample_idx = np.random.choice(len(X_test), size=2000, replace=False)
shap_values = explainer.shap_values(X_test[shap_sample_idx])

# Summary plot
fig, ax = plt.subplots(figsize=(10, 8))
shap.summary_plot(shap_values, X_test[shap_sample_idx], feature_names=feature_names, show=False)
plt.title("SHAP Feature Importance (LightGBM)")
plt.tight_layout()
plt.savefig(f"{ARTIFACT_DIR}/cem_shap_summary.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {ARTIFACT_DIR}/cem_shap_summary.png")

# Bar plot of mean absolute SHAP values
fig, ax = plt.subplots(figsize=(10, 8))
shap.summary_plot(shap_values, X_test[shap_sample_idx], feature_names=feature_names, 
                  plot_type="bar", show=False)
plt.title("Mean |SHAP| Value per Feature")
plt.tight_layout()
plt.savefig(f"{ARTIFACT_DIR}/cem_shap_bar.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {ARTIFACT_DIR}/cem_shap_bar.png")

## Phase 7: Save Winning Model & Register

In [ ]:
# Save LightGBM model
model_path = f"{MODEL_DIR}/cem_v3_lightgbm.joblib"
joblib.dump(final_model, model_path)
print(f"Saved model: {model_path} ({os.path.getsize(model_path)/1e6:.1f} MB)")

# Save GBR baseline too for comparison
joblib.dump(gbr, f"{MODEL_DIR}/cem_v3_gbr_baseline.joblib")
print(f"Saved baseline: {MODEL_DIR}/cem_v3_gbr_baseline.joblib")

# Save feature names for inference
joblib.dump(feature_names, f"{MODEL_DIR}/cem_v3_feature_names.joblib")
print(f"Saved feature names: {MODEL_DIR}/cem_v3_feature_names.joblib")

# Generate model card
model_card = f"""
# CEM Experience Score Model Card (v3.0)

## Model Details
- **Algorithm**: LightGBM (Gradient Boosting Decision Tree)
- **Version**: v3.0
- **Training Date**: {datetime.now().isoformat()}
- **Dataset**: Real BSS subscribers + OSS KPIs, March 2026
- **Samples**: {len(y):,} subscribers
- **Features**: {len(feature_names)} ({', '.join(feature_names)})

## Performance (Test Set)
- **R²**: {metrics_lgb_test['r2']:.4f}
- **MAE**: {metrics_lgb_test['mae']:.4f}
- **RMSE**: {metrics_lgb_test['rmse']:.4f}

## Hyperparameters
{final_params}

## Feature Importance (Top 5)
"""

# Add top 5 features by SHAP
mean_shap = np.abs(shap_values).mean(axis=0)
top5_idx = np.argsort(mean_shap)[-5:][::-1]
for i, idx in enumerate(top5_idx, 1):
    model_card += f"{i}. {feature_names[idx]} (|SHAP| = {mean_shap[idx]:.4f})\n"

with open(f"{MODEL_DIR}/cem_v3_model_card.md", "w") as f:
    f.write(model_card)
print(f"Saved model card: {MODEL_DIR}/cem_v3_model_card.md")

## Summary

| Metric | GBR (v2.0 baseline) | LightGBM (v3.0) | Improvement |
|--------|---------------------|-----------------|-------------|
| R² | {metrics_gbr_test['r2']:.4f} | {metrics_lgb_test['r2']:.4f} | +{metrics_lgb_test['r2']-metrics_gbr_test['r2']:.4f} |
| MAE | {metrics_gbr_test['mae']:.4f} | {metrics_lgb_test['mae']:.4f} | -{metrics_gbr_test['mae']-metrics_lgb_test['mae']:.4f} |
| RMSE | {metrics_gbr_test['rmse']:.4f} | {metrics_lgb_test['rmse']:.4f} | -{metrics_gbr_test['rmse']-metrics_lgb_test['rmse']:.4f} |

**Artifacts produced:**
- `models/cem_v3_lightgbm.joblib` — winning model
- `models/cem_v3_gbr_baseline.joblib` — baseline for comparison
- `models/cem_v3_feature_names.joblib` — feature list for inference
- `models/cem_v3_model_card.md` — documentation
- `data/cem_*_analysis.png` — evaluation plots

In [ ]:
print(f"[{datetime.now():%H:%M:%S}] CEM v3.0 Training Complete")